# Golden Set

In [11]:
import os
import re
import json
import pandas as pd
import numpy as np
import ollama as llm
from tqdm import tqdm

In [12]:
# File Paths
PROCESSED_DIR = "../data/processed" if os.path.exists("../data") else "data/processed"
RAW_DIR = "../data/raw" if os.path.exists("../data") else "data/raw"
SAMPLE_5K_PATH = os.path.join(PROCESSED_DIR, "amazon_help_pairs_sample_5k.csv")
SAMPLE_PATH = os.path.join(PROCESSED_DIR, "amazon_help_pairs_full.csv")
GOLDEN_JSON_PATH = os.path.join(PROCESSED_DIR, "golden_set_json.jsonl")
GOLDEN_CSV_PATH = os.path.join(PROCESSED_DIR, "golden_csv_set.csv")


In [13]:
df_5k = pd.read_csv(SAMPLE_5K_PATH)
df_5k.head()

,customer_tweet_id,customer_text,customer_clean,brand_tweet_id,brand_text,brand_clean,created_at
0,2602988,@AmazonHelp I'm not sure.i assumed they would ...,I'm not sure.i assumed they would be all the s...,2602989,@736628 I'm sorry about this. We want to be su...,I'm sorry about this. We want to be sure you a...,Mon Nov 27 14:12:39 +0000 2017
1,1087357,@AmazonHelp @376508 You really want to keep my...,You really want to keep my money? I'm waiting ...,1087358,@376509 I'm sorry for any trouble. Are you a s...,I'm sorry for any trouble. Are you a seller on...,Sun Oct 15 03:16:00 +0000 2017
2,2555183,It’s like the battery I ordered asked to be up...,It’s like the battery I ordered asked to be up...,2555182,@725590 We're always looking for ways to impro...,"We're always looking for ways to improve, Henr...",Thu Nov 16 15:54:28 +0000 2017
3,485273,@AmazonHelp A special amazon line will be cont...,A special amazon line will be contacting me af...,485275,@230457 Have you checked your emails to see if...,Have you checked your emails to see if there w...,Fri Dec 01 15:40:36 +0000 2017
4,607317,It's been 3 days since ordered sonething and ...,It's been 3 days since ordered sonething and I...,607315,@264327 We apologize for the inconvenience reg...,We apologize for the inconvenience regarding t...,Wed Nov 22 11:45:28 +0000 2017


In [14]:
# Heuristic rules to handle high-signal candidate pools
rules = {
    "ACCOUNT_PAYMENT_SECURITY": r"\b(charged|charge|unauthorized|stolen|card|bank|account|password|otp|gift card|fraud|hacked|scam)\b",
    "RETURN_REFUND": r"\b(refund|return|damaged|broken|exchange|pickup|money back|replace|replacement|wrong item)\b",
    "SERVICE_COMPLAINT": r"\b(worst|terrible|horrible|rude|driver|complaint|pathetic|court|legal|manager|supervisor|consumer court|disgusted)\b",
    "PRODUCT_POLICY_INQUIRY": r"\b(warranty|guarantee|policy|prime membership|renew|annual fee|available|restock|how do i)\b",
    "OUT_OF_SCOPE_CHITCHAT": r"\b(thanks|thank you|awesome|great|haha|cool|love amazon|kudos|nice)\b",
    "ORDER_TRACKING_DELAY": r"\b(delivery|delayed|late|tracking|status|courier|still not arrived|where is|dispatch|transit)\b"
}

# Adverserial cases
adversarial_rules = {
    "SARCASM": r"\b(thanks for nothing|great job amazon|wonderful delivery|love when you destroy)\b",
    "MULTILINGUAL": r"\b(que|por favor|donde|ayuda|pedido|hola)\b",
    "SHORT_VAGUE": r"^.{4,25}$" # Very short customer queries
}

In [15]:
selected_indices = set()
candidate_records = []

for intent, pattern in rules.items():
    subset = df_5k[
        df_5k["customer_clean"].str.contains(pattern, case = False, na = False) &
        (~df_5k.index.isin(selected_indices))
    ]
    sample_n = min(28, len(subset))
    sampled = subset.sample(n = sample_n, random_state = 42)
    selected_indices.update(sampled.index)
    
    for _, row in sampled.iterrows():
        candidate_records.append({
            "candidate_intent": intent,
            "difficulty": "standard",
            "customer_clean": row["customer_clean"],
            "brand_clean": row["brand_clean"]
        })

for edge_type, pattern in adversarial_rules.items():
    subset = df_5k[
        df_5k["customer_clean"].str.contains(pattern, case = False, na = False) &
        (~df_5k.index.isin(selected_indices))
    ]
    sample_n = min(10, len(subset))
    if sample_n > 0:
        sampled = subset.sample(n=sample_n, random_state=42)
        selected_indices.update(sampled.index)
        for _, row in sampled.iterrows():
            candidate_records.append({
                "candidate_intent": "AMBIGUOUS_EDGE_CASE",
                "difficulty": "adversarial_edge_case",
                "customer_clean": row["customer_clean"],
                "brand_clean": row["brand_clean"]
            })
df_candidates = pd.DataFrame(candidate_records).reset_index(drop=True)
print(f"[✔] Curated {len(df_candidates)} balanced candidate samples for labeling!")
print(df_candidates["candidate_intent"].value_counts())

C:\Users\Roschlynn\AppData\Local\Temp\ipykernel_34092\472846143.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_5k["customer_clean"].str.contains(pattern, case = False, na = False) &
C:\Users\Roschlynn\AppData\Local\Temp\ipykernel_34092\472846143.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_5k["customer_clean"].str.contains(pattern, case = False, na = False) &
C:\Users\Roschlynn\AppData\Local\Temp\ipykernel_34092\472846143.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_5k["customer_clean"].str.contains(pattern, case = False, na = False) &
C:\Users\Roschlynn\AppData\Local\Temp\ipykernel_34092\472846143.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To ac

[✔] Curated 188 balanced candidate samples for labeling!
candidate_intent
ACCOUNT_PAYMENT_SECURITY    28
RETURN_REFUND               28
SERVICE_COMPLAINT           28
PRODUCT_POLICY_INQUIRY      28
OUT_OF_SCOPE_CHITCHAT       28
ORDER_TRACKING_DELAY        28
AMBIGUOUS_EDGE_CASE         20
Name: count, dtype: int64


In [16]:
MODEL = "llama3.1:8b"
SYSTEM_PROMPT = """
    You are an expert customer support quality auditor for AmazonHelp.
    Analyze this customer tweet and provide ground-truth labeling.
    You need to choose exactly one of the following intents:
    1. ORDER_TRACKING_DELAY (delivery delays, courier status, missing packages)
    2. RETURN_REFUND (item return, pickups, refund status, replacement)
    3. ACCOUNT_PAYMENT_SECURITY (unauthorized charges, card issues, account locks, OTP, scams)
    4. PRODUCT_POLICY_INQUIRY (warranty, fees, prime rules, product availability)
    5. SERVICE_COMPLAINT (problematic drivers, service failures, legal threats, angry customers)
    6. OUT_OF_SCOPE (general praise, greetings, buyers remorse, banters)

    The escalation policy is as follows
    - should_escalate = True
        1. Payment/Account security risk (fraud, unauthorized charges)
        2. Legal Threats, Regulatory Threats, Consumer Court Threats
        3. Extremely angry customers, abusive threats, persistent service failures
        4. General feedback and recommedations
    
    Respond only with valid JSON in the following format:
    {
        "gold_intent": "<INTENT_NAME>",
        "gold_should_escalate": <true or false>,
        "gold_escalation_reason": "<Precise and Concise 1 sentence reason if true else null>",
        "difficulty": "<standard OR edge_case OR high_risk>"
    }
"""

In [17]:
labeled_samples = []

print(f"[*] Bootstrapping labels for {len(df_candidates)} samples using Ollama ({MODEL})...")
for idx, row in tqdm(df_candidates.iterrows(), total = len(df_candidates)):
    prompt = f'Customer Tweet: "{row["customer_clean"]}"'
    
    try:
        response = llm.chat(
            model = MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            format="json",
            options={"temperature": 0.0}
        )
        parsed = json.loads(response["message"]["content"])
    except Exception as e:
        print(f"Exception: {e} has occured")
    # Ensure difficulty captures high risk
    if parsed.get("gold_should_escalate"):
        difficulty = "high_risk" if parsed.get("gold_intent") in ["ACCOUNT_PAYMENT_SECURITY", "SERVICE_COMPLAINT"] else "standard"
    else:
        difficulty = row["difficulty"]
    labeled_samples.append({
        "id": f"gold_{idx+1:03d}",
        "customer_tweet": row["customer_clean"],
        "reference_reply": row["brand_clean"],
        "gold_intent": parsed.get("gold_intent", "ORDER_TRACKING_DELAY"),
        "gold_should_escalate": bool(parsed.get("gold_should_escalate", False)),
        "gold_escalation_reason": parsed.get("gold_escalation_reason") if parsed.get("gold_should_escalate") else None,
        "difficulty": difficulty
    })
df_golden = pd.DataFrame(labeled_samples)
print("\n[✔] Labeling complete!")
print("\n--- Intent Distribution ---")
print(df_golden["gold_intent"].value_counts())
print("\n--- Escalation Breakdown ---")
print(df_golden["gold_should_escalate"].value_counts(normalize=True).mul(100).round(1).astype(str) + "%")
    


[*] Bootstrapping labels for 188 samples using Ollama (llama3.1:8b)...


100%|██████████| 188/188 [40:11<00:00, 12.83s/it]


[✔] Labeling complete!

--- Intent Distribution ---
gold_intent
ORDER_TRACKING_DELAY        46
PRODUCT_POLICY_INQUIRY      37
SERVICE_COMPLAINT           36
RETURN_REFUND               28
ACCOUNT_PAYMENT_SECURITY    21
OUT_OF_SCOPE                20
Name: count, dtype: int64

--- Escalation Breakdown ---
gold_should_escalate
False    74.5%
True     25.5%
Name: proportion, dtype: str


In [18]:
# Inspect 5 escalated examples to verify policy correctness
print("Sample Escalations Review:")
escalations = df_golden[df_golden["gold_should_escalate"] == True].sample(min(5, len(df_golden)), random_state=42)

for _, row in escalations.iterrows():
    print(f"[{row['id']}] Intent: {row['gold_intent']}")
    print(f"Tweet: {row['customer_tweet']}")
    print(f"Reason: {row['gold_escalation_reason']}")
    print("-" * 60)

# Save to both JSONL (for evaluation harness) and CSV (for quick human inspection)
df_golden.to_json(GOLDEN_JSON_PATH, orient="records", lines=True)
df_golden.to_csv(GOLDEN_CSV_PATH, index=False)

print(f"\n[✔] SUCCESS: Golden set ({len(df_golden)} examples) exported to:")
print(f"    -> JSONL: {GOLDEN_JSON_PATH}")
print(f"    -> CSV:   {GOLDEN_CSV_PATH}")


Sample Escalations Review:
[gold_071] Intent: SERVICE_COMPLAINT
Tweet: are TERRIBLE, the customer service department are even worse, and furthermore if you want next day delivery with PRIME you wont get it. You call up and you get bullshitted over the phone TBH, they cant even understand your language!
Reason: Customer is extremely angry and reports persistent service failures, including unhelpful customer service representatives.
------------------------------------------------------------
[gold_105] Intent: ACCOUNT_PAYMENT_SECURITY
Tweet: One more day has passed without any updates. Why is it so difficult to unlock my account? If you can’t unlock, refund my prime membership amount. ? #nothingprimeinamazonprime
Reason: Customer is experiencing account lock and requesting refund due to account security issue
------------------------------------------------------------
[gold_070] Intent: RETURN_REFUND
Tweet: #AmazonIndia worst service. Almost a month has passed but my refund still not p